# dfguard — pandas quickstart

This notebook demonstrates the full dfguard pandas API end-to-end.

> **Note:** `arm()` patches all annotated functions in a module at import time and does not work inside notebooks. Use `@dfg.enforce` explicitly per function — that is the correct approach for notebook usage.

**Requirements:** `pip install 'dfguard[pandas]' pyarrow`

In [1]:
import numpy as np
import pandas as pd
import pyarrow as pa

import dfguard.pandas as dfg
from dfguard.pandas import Optional

## 1. Schema definition

Two ways to declare a schema: class-based declaration using `PandasSchema`, and `schema_of()` which captures the schema from a live DataFrame.

Fields use **assignment form** (`=`) to avoid Pylance `reportInvalidTypeForm` warnings. Annotation form (`:`) also works.

`Optional[dtype]` marks a field as intentionally nullable. dfguard checks that the column exists with the correct dtype but does NOT enforce null presence or absence.

In [2]:
# Class-based schema declaration (assignment form)
# Nested type: pd.ArrowDtype with pyarrow gives full nested-type enforcement,
# the same level as PySpark and Polars.
class OrderSchema(dfg.PandasSchema):
    order_id   = np.dtype("int64")
    customer   = pd.StringDtype()
    amount     = np.dtype("float64")
    active     = pd.BooleanDtype()
    # ArrowDtype: list of structs — requires pyarrow
    line_items = pd.ArrowDtype(pa.list_(pa.struct([
        pa.field("sku",      pa.string()),
        pa.field("quantity", pa.int32()),
        pa.field("price",    pa.float64()),
    ])))
    zip_code   = Optional[pd.StringDtype()]   # nullable: dtype enforced, nulls allowed

print("Schema fields:", list(OrderSchema.to_dtype_dict().keys()))

Schema fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']


In [3]:
# Build a matching DataFrame
line_items_data = [
    [{"sku": "A1", "quantity": 2, "price": 9.99}, {"sku": "B2", "quantity": 1, "price": 4.50}],
    [{"sku": "C3", "quantity": 3, "price": 14.00}],
]
orders_df = pd.DataFrame({
    "order_id":   pd.array([101, 102], dtype="int64"),
    "customer":   pd.array(["Alice", "Bob"], dtype="string"),
    "amount":     pd.array([24.48, 14.00], dtype="float64"),
    "active":     pd.array([True, False], dtype="boolean"),
    "line_items": pd.array(line_items_data, dtype=pd.ArrowDtype(pa.list_(pa.struct([
        pa.field("sku",      pa.string()),
        pa.field("quantity", pa.int32()),
        pa.field("price",    pa.float64()),
    ])))),
    "zip_code":   pd.array(["94107", None], dtype="string"),
})
orders_df

,order_id,customer,amount,active,line_items,zip_code
0,101,Alice,24.48,True,"[{'sku': 'A1', 'quantity': 2, 'price': 9.99} ...",94107
1,102,Bob,14.00,False,"[{'sku': 'C3', 'quantity': 3, 'price': 14.0}]",<NA>


In [4]:
# schema_of() captures an exact schema from a live DataFrame.
# The resulting type requires an exact column match (no extras).
CapturedSchema = dfg.schema_of(orders_df)
print("schema_of captured:", CapturedSchema)
print("isinstance check (exact match):", isinstance(orders_df, CapturedSchema))

schema_of captured: <class 'dfguard.pandas.dataset.SchemaType'>
isinstance check (exact match): True


## 2. Schema inheritance

In [5]:
# EnrichedOrderSchema inherits all fields from OrderSchema and adds revenue.
class EnrichedOrderSchema(OrderSchema):
    revenue = np.dtype("float64")

print("Enriched fields:", list(EnrichedOrderSchema.to_dtype_dict().keys()))
# All parent fields are present plus 'revenue'
assert "order_id"  in EnrichedOrderSchema.to_dtype_dict()
assert "revenue"   in EnrichedOrderSchema.to_dtype_dict()
print("Inheritance confirmed.")

Enriched fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']
Inheritance confirmed.


## 3. `@dfg.enforce` decorator

Decorate a function with `@dfg.enforce` to validate schema at call time. The decorator reads type annotations on parameters and checks them against the incoming DataFrame.

Two modes:
- `@dfg.enforce` (default): `subset=True` — the DataFrame must contain at least the declared columns; extra columns are fine.
- `@dfg.enforce(subset=False)`: the DataFrame must match the declared columns exactly.

In [6]:
# subset=True (default): extra columns on the DataFrame are fine.
@dfg.enforce
def compute_revenue(df: OrderSchema) -> pd.DataFrame:
    return df.assign(revenue=df["amount"] * 1.1)

enriched_df = compute_revenue(orders_df)
print("compute_revenue passed, columns:", list(enriched_df.columns))

compute_revenue passed, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']


In [7]:
# subset=False: exact match — the DataFrame must have ONLY the declared columns.
@dfg.enforce(subset=False)
def write_final(df: OrderSchema) -> None:
    print(f"write_final called with {len(df)} rows, columns: {list(df.columns)}")

# PASSING: orders_df has exactly the OrderSchema columns.
write_final(orders_df)

# FAILING: enriched_df has an extra 'revenue' column.
try:
    write_final(enriched_df)
except TypeError as e:
    print(f"\nCaught expected error:\n{e}")

write_final called with 2 rows, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']

Caught expected error:
Schema mismatch in write_final() argument 'df':
  expected: OrderSchema
  received: order_id:int64, customer:string, amount:float64, active:boolean, line_items:list<item: struct<sku: string, quantity: int32, price: double>>[pyarrow], zip_code:string, revenue:float64


In [8]:
# FAILING: wrong dtype — amount as string instead of float64.
bad_df = orders_df.copy()
bad_df["amount"] = bad_df["amount"].astype(str)

try:
    compute_revenue(bad_df)
except TypeError as e:
    print(f"Caught expected error:\n{e}")

Caught expected error:
Schema mismatch in compute_revenue() argument 'df':
  expected: OrderSchema
  received: order_id:int64, customer:string, amount:object, active:boolean, line_items:list<item: struct<sku: string, quantity: int32, price: double>>[pyarrow], zip_code:string


## 4. `assert_valid` for load-time validation

In [9]:
# assert_valid raises SchemaValidationError if the DataFrame does not match.
# Use this at load time (e.g. after reading a CSV or Parquet file).

# PASSING
OrderSchema.assert_valid(orders_df)
print("assert_valid passed.")

# PASSING with subset=True (default): extra columns are fine.
OrderSchema.assert_valid(enriched_df, subset=True)
print("assert_valid(subset=True) passed on DataFrame with extra column.")

# FAILING with subset=False: enriched_df has 'revenue' which is not in OrderSchema.
try:
    OrderSchema.assert_valid(enriched_df, subset=False)
except dfg.SchemaValidationError as e:
    print(f"\nCaught expected error:\n{e}")

assert_valid passed.
assert_valid(subset=True) passed on DataFrame with extra column.

Caught expected error:
Schema validation failed:
  ✗ Unexpected column 'revenue' (strict mode)

Schema evolution (most recent last):
  [input]  {order_id:int64, customer:string, amount:float64, active:boolean, line_items:list<item: struct<sku: string, quantity: int32, price: double>>[pyarrow], zip_code:string, revenue:float64}  (no schema change)


## 5. Schema utilities: `from_struct`, `to_code`, `diff`, `empty`

In [10]:
# from_struct: generate a PandasSchema from a dtype dict (e.g. from df.dtypes).
# Useful for capturing schemas at runtime without writing a class by hand.
InferredSchema = dfg.PandasSchema.from_struct(dict(orders_df.dtypes), name="InferredOrderSchema")
print("from_struct fields:", list(InferredSchema.to_dtype_dict().keys()))

from_struct fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']


In [11]:
# to_code: generate Python source for the schema class.
print(InferredSchema.to_code())

import numpy as np
import pandas as pd
from dfguard.pandas import PandasSchema, Optional

class InferredOrderSchema(PandasSchema):
    order_id: np.dtype('int64')
    customer: pd.StringDtype()
    amount: np.dtype('float64')
    active: pd.BooleanDtype()
    line_items: pd.ArrowDtype()
    zip_code: pd.StringDtype()


In [12]:
# diff: show differences between two schemas.
print(OrderSchema.diff(EnrichedOrderSchema))
print()
# Identical schemas produce a message saying so.
print(OrderSchema.diff(OrderSchema))

Diff OrderSchema -> EnrichedOrderSchema:
  Missing column 'revenue' (expected float64)

OrderSchema and OrderSchema are identical


In [13]:
# empty: return a zero-row DataFrame with the correct dtypes.
# Useful for initialising output buffers or test fixtures.
empty_df = OrderSchema.empty()
print("empty() shape:", empty_df.shape)
print("empty() dtypes:\n", empty_df.dtypes)

empty() shape: (0, 6)
empty() dtypes:
 order_id                                                  int64
customer                                         string[python]
amount                                                  float64
active                                                  boolean
line_items    list<item: struct<sku: string, quantity: int32...
zip_code                                         string[python]
dtype: object


## 6. Schema history via `dfg.dataset()`

`dfg.dataset()` wraps a DataFrame in a tracking object. Each mutating operation (assign, rename, drop, ...) records a schema change in an immutable history log.

In [14]:
ds = dfg.dataset(orders_df)

# Each step returns a new dataset with the updated history.
ds2 = ds.assign(revenue=ds._df["amount"] * 1.1)
ds3 = ds2.rename(columns={"customer": "customer_name"})
ds4 = ds3.drop("zip_code")

# Print the full schema evolution.
ds4.schema_history.print()

────────────────────────────────────────────────────────────
Schema Evolution
────────────────────────────────────────────────────────────
  [ 0] input
       {order_id:int64, customer:string, amount:float64, active:boolean, line_items:list<item: struct<sku: string, quantity: int32, price: double>>[pyarrow], zip_code:string}  (no schema change)
  [ 1] assign(['revenue'])
       added: revenue
  [ 2] rename({'customer': 'customer_name'})
       added: customer_name | dropped: customer
  [ 3] drop(['zip_code'])
       dropped: zip_code
────────────────────────────────────────────────────────────


## 7. Validate and check error messages (both passing and failing)

In [15]:
# validate() returns a list of errors without raising.
# PASSING: empty list
errors = OrderSchema.validate(orders_df)
print("Errors on valid DataFrame:", errors)

# FAILING: missing column and wrong dtype
broken_df = orders_df.drop(columns=["order_id"]).copy()
broken_df["amount"] = broken_df["amount"].astype(str)
errors = OrderSchema.validate(broken_df)
print("\nErrors on broken DataFrame:")
for err in errors:
    print(" -", err)

Errors on valid DataFrame: []

Errors on broken DataFrame:
 - Missing column 'order_id' (expected int64)
 - Column 'amount': type mismatch: expected float64, got object


In [16]:
# FAILING: nested type mismatch — wrong element type inside the ArrowDtype.
wrong_items_dtype = pd.ArrowDtype(pa.list_(pa.struct([
    pa.field("sku",      pa.string()),
    pa.field("quantity", pa.int64()),   # int64 instead of int32
    pa.field("price",    pa.float64()),
])))
nested_wrong_df = orders_df.copy()
nested_wrong_df["line_items"] = nested_wrong_df["line_items"].astype(wrong_items_dtype)

errors = OrderSchema.validate(nested_wrong_df)
print("Nested type errors:")
for err in errors:
    print(" -", err)

Nested type errors:
 - Column 'line_items': type mismatch: expected list<item: struct<sku: string, quantity: int32, price: double>>[pyarrow], got list<item: struct<sku: string, quantity: int64, price: double>>[pyarrow]


## 8. Putting it together: a typed pipeline

A minimal pipeline that uses `@dfg.enforce` on each stage with different schemas.

In [17]:
class ReportSchema(EnrichedOrderSchema):
    customer_name = pd.StringDtype()

@dfg.enforce
def enrich(df: OrderSchema) -> pd.DataFrame:
    return df.assign(
        revenue=df["amount"] * 1.1,
        customer_name=df["customer"],
    )

@dfg.enforce
def summarise(df: ReportSchema) -> pd.DataFrame:
    return df[["order_id", "customer_name", "revenue"]]

result = summarise(enrich(orders_df))
print(result)

   order_id customer_name  revenue
0       101         Alice   26.928
1       102           Bob   15.400
